In [1]:
from pathlib import Path
import pandas as pd

file_path = Path("wiki-RfA.txt")
text = file_path.read_text(encoding="utf-8", errors="replace")
records = text.strip().split("\n\n")

fields = ["SRC", "TGT", "VOT", "RES", "YEA", "DAT", "TXT"]
parsed = []

for block in records:
    row = {k: None for k in fields}
    current_key = None
    current_value = []

    for line in block.splitlines():
        matched = False
        for key in fields:
            prefix = f"{key}:"
            if line.startswith(prefix):
                if current_key is not None:
                    row[current_key] = "\n".join(current_value).strip()
                current_key = key
                current_value = [line[len(prefix):].strip()]
                matched = True
                break
        
        if not matched and current_key is not None:
            current_value.append(line)

    if current_key is not None:
        row[current_key] = "\n".join(current_value).strip()

    parsed.append(row)

df = pd.DataFrame(parsed)

print(f"Number of record blocks: {len(records)}")
print("\nFirst raw block:\n")
print(records[0])

print("\nParsed DataFrame info:\n")
print(df.head())
print(df.shape)
print(df.columns.tolist())

Number of record blocks: 198275

First raw block:

SRC:Steel1943
TGT:BDD
VOT:1
RES:1
YEA:2013
DAT:23:13, 19 April 2013
TXT:'''Support''' as co-nom.

Parsed DataFrame info:

          SRC  TGT VOT RES   YEA                   DAT  \
0   Steel1943  BDD   1   1  2013  23:13, 19 April 2013   
1  Cuchullain  BDD   1   1  2013  01:04, 20 April 2013   
2   INeverCry  BDD   1   1  2013  23:43, 19 April 2013   
3   Cncmaster  BDD   1   1  2013  00:11, 20 April 2013   
4  Miniapolis  BDD   1   1  2013  00:56, 20 April 2013   

                                                 TXT  
0                           '''Support''' as co-nom.  
1                      '''Support''' as nominator.--  
2                            '''Support''' per noms.  
3  '''Support''' per noms. BDD is a strong contri...  
4  '''Support''', with great pleasure. I work wit...  
(198275, 7)
['SRC', 'TGT', 'VOT', 'RES', 'YEA', 'DAT', 'TXT']


In [2]:
import networkx as nx

# keep only rows where source and target exist
df_graph = df.dropna(subset=["SRC", "TGT"]).copy()

# create directed graph
G = nx.from_pandas_edgelist(
    df_graph,
    source="SRC",
    target="TGT",
    edge_attr=["VOT", "RES", "YEA", "DAT", "TXT"],
    create_using=nx.DiGraph()
)

print(G)
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")

DiGraph with 11381 nodes and 189003 edges
Number of nodes: 11381
Number of edges: 189003


In [3]:
import json
from networkx.readwrite import json_graph

graph_data = json_graph.node_link_data(G)

with open("wiki_rfa_graph.json", "w", encoding="utf-8") as f:
    json.dump(graph_data, f, ensure_ascii=False)

print("Graph saved to wiki_rfa_graph.json")

Graph saved to wiki_rfa_graph.json
